In [ ]:
!pip install nltk

In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
import nltk

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

fake = pd.read_csv("/content/drive/MyDrive/Fake.csv")
true = pd.read_csv("/content/drive/MyDrive/True.csv")

In [ ]:
fake["label"] = 0
true["label"] = 1

data = pd.concat([fake,true])
data = data.sample(frac=1)

data = data[["text","label"]]

In [ ]:
data = data.sample(20000)

In [ ]:
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = text.lower()
    text = re.sub("[^a-zA-Z]"," ",text)

    words = text.split()
    words = [w for w in words if w not in stop_words]

    return " ".join(words)

data["text"] = data["text"].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(data["text"])

X = tokenizer.texts_to_sequences(data["text"])

X = pad_sequences(X,maxlen=500)

y = data["label"]

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2
)

In [ ]:
model = Sequential()

model.add(Embedding(5000,64,input_length=500))

model.add(Bidirectional(LSTM(64)))

model.add(Dense(1,activation="sigmoid"))

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.fit(
    X_train,
    y_train,
    epochs=3,
    batch_size=64,
    validation_data=(X_test,y_test)
)

Epoch 1/3
250/250 ━━━━━━━━━━━━━━━━━━━━ 195s 749ms/step - accuracy: 0.9369 - loss: 0.1784 - val_accuracy: 0.9790 - val_loss: 0.0678
Epoch 2/3
250/250 ━━━━━━━━━━━━━━━━━━━━ 202s 748ms/step - accuracy: 0.9880 - loss: 0.0437 - val_accuracy: 0.9852 - val_loss: 0.0524
Epoch 3/3
250/250 ━━━━━━━━━━━━━━━━━━━━ 202s 747ms/step - accuracy: 0.9888 - loss: 0.0382 - val_accuracy: 0.9725 - val_loss: 0.0920


In [ ]:
import tensorflow as tf
print(tf.__version__)

2.19.0


In [ ]:
# save weights only
model.save_weights("model.weights.h5")

In [ ]:
with open("tokenizer.pkl","wb") as f:
    pickle.dump(tokenizer,f)

In [ ]:
from google.colab import files
files.download("tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("model.weights.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>